In [1]:
# amt.py — Anticipatory Music Transformer wrapper
from transformers import AutoModelForCausalLM
from anticipation import ops
from anticipation.convert import midi_to_events, events_to_midi
from anticipation.tokenize import extract_instruments
from anticipation.sample import generate
import torch

MODELS = {
    "small":  "stanford-crfm/music-small-800k",
    "medium": "stanford-crfm/music-medium-800k",
    "large":  "stanford-crfm/music-large-800k",
}

def load(size="small", device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(MODELS[size], torch_dtype=dtype).to(device).eval()
    return model

def continue_midi(model, midi_path, out_path, prompt_seconds=5, generate_seconds=20, top_p=0.98):
    """Feed a MIDI file as a prompt, generate a continuation, save result."""
    events = midi_to_events(midi_path)
    history = ops.clip(events, 0, prompt_seconds, clip_duration=False)
    new_events = generate(model, prompt_seconds, prompt_seconds + generate_seconds,
                          inputs=history, top_p=top_p)
    events_to_midi(new_events).save(out_path)

def accompany_melody(model, midi_path, out_path, melody_program=53,
                    prompt_seconds=5, generate_seconds=20, top_p=0.98):
    """Extract a melody track, generate accompaniment around it, save combined MIDI."""
    events = midi_to_events(midi_path)
    events, melody = extract_instruments(events, [melody_program])
    history = ops.clip(events, 0, prompt_seconds, clip_duration=False)
    accompaniment = generate(model, prompt_seconds, prompt_seconds + generate_seconds,
                             inputs=history, controls=melody, top_p=top_p)
    combined = ops.combine(accompaniment, melody)
    events_to_midi(combined).save(out_path)